<a href="https://colab.research.google.com/github/Prajwaln08/SQLSmith/blob/main/07_SFT_Dataset_Preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SQLSmith — 07 | SFT Dataset Preparation
**Milestone B · CPU only · Spider 1.0 training data**

Creates frozen, database-disjoint training and internal validation sets for **Qwen2.5-Coder-1.5B / 7B QLoRA**. No fine-tuning, GPU, or frontier-model distillation happens here.

**Separation rule:** Spider `dev.json` (1,034 examples) is held out for *official external evaluation*. All SFT examples come from `train_spider.json` (and optionally `train_others.json`). An internal validation split is chosen by database, never by individual question. We check for any overlap with development database IDs and exclude offending records.

**Outputs:** Drive `.../SQLSmith/data/processed/sft_v1/` — JSONL chat examples, flat QA records, quality checks, manifest, and deterministic DB split. **Do not publish raw database files or private data.** Check Spider's CC-BY-SA terms before redistributing training-derived artifacts.


## 1 — Install lightweight dependencies

In [1]:
%pip -q install "huggingface_hub>=0.28" "pandas>=2.0" "sqlglot>=25"

## 2 — Mount Drive and define fixed paths

In [2]:
from google.colab import drive
from pathlib import Path
import json, hashlib, random, sys, platform, sqlite3, time, collections, csv
import pandas as pd

drive.mount('/content/drive')
PROJECT = Path('/content/drive/MyDrive/Prajwal/Projects/SQLSmith')
DATA = PROJECT / 'data'
OUTPUT = DATA / 'processed' / 'sft_v1'
OUTPUT.mkdir(parents=True, exist_ok=True)
RAW_ROOT = Path('/content/sqlsmith_spider_source')
RAW_ROOT.mkdir(parents=True, exist_ok=True)
SEED = 42
VALIDATION_DB_FRACTION = 0.10
INCLUDE_OTHERS = True  # Spider's supplemental train_others.json, when present
VERIFY_SQL_COMPILATION = True  # Gold SQL only, on public benchmark DBs
SQL_TIMEOUT_SECONDS = 4
print('Dataset outputs:', OUTPUT)


Mounted at /content/drive
Dataset outputs: /content/drive/MyDrive/Prajwal/Projects/SQLSmith/data/processed/sft_v1


## 3 — Acquire the public Spider 1.0 archive
Reuse the same mirrored archive used in Notebooks 01–03. The download is a benchmark artifact, **not** an enterprise schema.

In [3]:
from huggingface_hub import hf_hub_download
import zipfile
ARCHIVE_REPO = 'HAL-9001/spider-databases'
ARCHIVE_NAME = 'spider_data.zip'
EXPECTED_ARCHIVE_SHA256 = '00636695dabed6b5f4b8328a16b13e069a2f16591d5efcce57660669c85b121b'
archive = Path(hf_hub_download(repo_id=ARCHIVE_REPO, repo_type='dataset', filename=ARCHIVE_NAME))
def sha256_file(p):
    h = hashlib.sha256()
    with Path(p).open('rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()
archive_hash = sha256_file(archive)
assert archive_hash == EXPECTED_ARCHIVE_SHA256, f'Unexpected Spider archive SHA256: {archive_hash}'
with zipfile.ZipFile(archive) as z:
    root = RAW_ROOT.resolve()
    for member in z.infolist():
        dest = (root / member.filename).resolve()
        if not dest.is_relative_to(root):
            raise ValueError(f'Unsafe archive member: {member.filename}')
    z.extractall(root)
SPIDER = RAW_ROOT / 'spider_data'
print('Spider extracted:', SPIDER)
print('Files:', sorted(p.name for p in SPIDER.glob('*.json')))


spider_data.zip: reconstructing file:   0%|          |  0.00B /  206MB            

spider_data.zip: downloading bytes:           |  0.00B            

Spider extracted: /content/sqlsmith_spider_source/spider_data
Files: ['dev.json', 'tables.json', 'test.json', 'test_tables.json', 'train_others.json', 'train_spider.json']


## 4 — Load original training, dev and schema metadata

In [4]:
def load_json(p):
    with Path(p).open(encoding='utf-8') as f: return json.load(f)
for name in ['train_spider.json','dev.json','tables.json']:
    assert (SPIDER / name).exists(), f'Missing original file: {name}'
train_primary = load_json(SPIDER/'train_spider.json')
extra_path = SPIDER/'train_others.json'
train_extra = load_json(extra_path) if (INCLUDE_OTHERS and extra_path.exists()) else []
dev = load_json(SPIDER/'dev.json')
tables = {x['db_id']: x for x in load_json(SPIDER/'tables.json')}
DB_ROOT = SPIDER / 'database'
assert DB_ROOT.exists()
print('train_spider:',len(train_primary),'train_others:',len(train_extra),'dev:',len(dev),'schema DBs:',len(tables))
assert len(dev) == 1034, 'Unexpected Spider dev size — investigate dataset version'


train_spider: 7000 train_others: 1659 dev: 1034 schema DBs: 166


## 5 — Quarantine split leakage and missing metadata
We never train on the development examples or a development database. We keep an explicit audit of exclusions.

In [5]:
dev_dbs = {str(x['db_id']) for x in dev}
dev_qas = {(str(x['db_id']),str(x['question']).strip(),str(x['query']).strip()) for x in dev}
source_records = [(x,'train_spider') for x in train_primary] + [(x,'train_others') for x in train_extra]
accepted, excluded = [], []
for source_index,(example, origin) in enumerate(source_records):
    db_id=str(example.get('db_id',''))
    question=str(example.get('question','')).strip()
    sql=str(example.get('query','')).strip()
    reason = None
    if not db_id or not question or not sql: reason='missing_required_fields'
    elif db_id in dev_dbs: reason='dev_database_overlap'
    elif (db_id,question,sql) in dev_qas: reason='dev_example_overlap'
    elif db_id not in tables: reason='no_schema_metadata'
    elif not (DB_ROOT/db_id/f'{db_id}.sqlite').exists(): reason='missing_sqlite_db'
    row={'source_index':source_index,'origin':origin,'db_id':db_id,'question':question,'gold_sql':sql}
    if reason: excluded.append({**row,'reason':reason})
    else: accepted.append(row)
print('Accepted before duplicate/SQL checks:',len(accepted),'Excluded:',len(excluded))
print('Exclusion reasons:',dict(collections.Counter(e['reason'] for e in excluded)))
assert accepted, 'No eligible training examples'
assert not ({x['db_id'] for x in accepted} & dev_dbs), 'Development database overlap detected'


Accepted before duplicate/SQL checks: 8659 Excluded: 0
Exclusion reasons: {}


## 6 — Schema formatting matched to Notebook 01/06 full-schema prompts
Use original table/column names and foreign key relationships. This is intentionally **full schema**, not schema linking.

In [6]:
def spider_schema_text(db_id):
    meta=tables[db_id]
    tabnames=meta['table_names_original']
    cols=meta['column_names_original']
    types=meta['column_types']
    result=[]
    for ti,tname in enumerate(tabnames):
        table_cols=[f'  {cname} {types[ci]}' for ci,(idx,cname) in enumerate(cols) if idx==ti]
        result.append('Table: '+tname+'\n'+'\n'.join(table_cols))
    result.append('\nForeign keys:')
    for a,b in meta['foreign_keys']:
        t1,c1=cols[a];t2,c2=cols[b]
        if t1>=0 and t2>=0:
            result.append(f'{tabnames[t1]}.{c1} = {tabnames[t2]}.{c2}')
    return '\n\n'.join(result)

SYSTEM_PROMPT = 'You are an expert text-to-SQL assistant. Generate exactly one valid SQLite SELECT query. Use only the tables and columns in the schema. Return SQL only, without markdown or explanation.'
def make_messages(question, schema, gold_sql):
    return [
        {'role':'system','content':SYSTEM_PROMPT},
        {'role':'user','content':f'Schema:\n{schema}\n\nQuestion: {question}'},
        {'role':'assistant','content':gold_sql}
    ]
print(spider_schema_text(accepted[0]['db_id'])[:1200])


Table: department
  Department_ID number
  Name text
  Creation text
  Ranking number
  Budget_in_Billions number
  Num_Employees number

Table: head
  head_ID number
  name text
  born_state text
  age number

Table: management
  department_ID number
  head_ID number
  temporary_acting text


Foreign keys:

management.head_ID = head.head_ID

management.department_ID = department.Department_ID


## 7 — Deduplicate exact records
Keep semantic variants and distinct gold SQL; remove only identical `(db_id, question, SQL)` tuples.

In [7]:
seen=set(); unique=[]; duplicates=[]
for row in accepted:
    k=(row['db_id'],row['question'],row['gold_sql'])
    if k in seen: duplicates.append(row)
    else: seen.add(k); unique.append(row)
print('Unique:',len(unique),'Exact duplicate rows:',len(duplicates))


Unique: 8651 Exact duplicate rows: 8


## 8 — Validate gold SQL parses and compiles in read-only SQLite
`EXPLAIN QUERY PLAN` checks basic SQLite column and table resolution without fetching result rows. Some valid dataset statements may be unsupported by the local SQLite build; we **quarantine**, do not silently rewrite, those examples.

In [8]:
import sqlglot
from sqlglot import exp
REJECTION_COUNTS=collections.Counter()
def check_gold(db_id,sql):
    try:
        trees=sqlglot.parse(sql,read='sqlite')
        if len(trees)!=1 or trees[0] is None: return 'not_one_statement'
        if not isinstance(trees[0],(exp.Select,exp.Union,exp.Intersect,exp.Except)):
            return 'non_query_root'
    except Exception: return 'parse_failed'
    if VERIFY_SQL_COMPILATION:
        path=(DB_ROOT/db_id/f'{db_id}.sqlite').resolve()
        conn=sqlite3.connect(path.as_uri()+'?mode=ro',uri=True,timeout=SQL_TIMEOUT_SECONDS)
        try:
            conn.execute('PRAGMA query_only=ON')
            deadline=time.monotonic()+SQL_TIMEOUT_SECONDS
            conn.set_progress_handler(lambda: int(time.monotonic()>deadline),1000)
            conn.execute('EXPLAIN QUERY PLAN '+sql).fetchall()
        except sqlite3.Error as e: return 'sqlite_compile_failed: '+str(e)[:120]
        finally: conn.close()
    return None

valid=[]; rejected=[]
for row in unique:
    reason=check_gold(row['db_id'],row['gold_sql'])
    if reason:
        rejected.append({**row,'reason':reason}); REJECTION_COUNTS[reason.split(':')[0]]+=1
    else:valid.append(row)
print('Validated:',len(valid),'Quarantined:',len(rejected))
print('Reasons:',dict(REJECTION_COUNTS))
assert len(valid)>0


Validated: 8648 Quarantined: 3
Reasons: {'sqlite_compile_failed': 3}


## 9 — Freeze a database-disjoint internal validation split
Select approximately 10% of **training database IDs** for validation, seed 42. Because DB sizes differ, the fraction of examples may differ from 10%. Spider dev remains untouched.

In [9]:
db_ids=sorted({r['db_id'] for r in valid})
assert len(db_ids)>1, 'Need multiple training DBs for database-disjoint validation'
rng=random.Random(SEED); shuffled=db_ids[:];rng.shuffle(shuffled)
n_val=max(1,round(VALIDATION_DB_FRACTION*len(db_ids)))
val_dbs=set(shuffled[:n_val]); train_dbs=set(shuffled[n_val:])
train_rows=[r for r in valid if r['db_id'] in train_dbs]
val_rows=[r for r in valid if r['db_id'] in val_dbs]
assert train_rows and val_rows
assert train_dbs.isdisjoint(val_dbs)
assert (train_dbs|val_dbs).isdisjoint(dev_dbs)
print('Training DBs:',len(train_dbs),'Validation DBs:',len(val_dbs),'Dev DBs:',len(dev_dbs))
print('Training examples:',len(train_rows),'Internal validation examples:',len(val_rows))
print('Validation IDs:',sorted(val_dbs))


Training DBs: 131 Validation DBs: 15 Dev DBs: 20
Training examples: 7931 Internal validation examples: 717
Validation IDs: ['club_1', 'cre_Drama_Workshop_Groups', 'customers_and_products_contacts', 'election', 'icfp_1', 'insurance_and_eClaims', 'insurance_policies', 'manufacturer', 'match_season', 'school_bus', 'soccer_1', 'student_assessment', 'tracking_orders', 'twitter_1', 'wine_1']


## 10 — Render supervised chat records
Each record includes `messages` with assistant gold SQL; the future trainer will mask system/user tokens and optimize only the assistant answer. We also export simple flat records for inspection.

In [10]:
def add_formats(rows):
    chats=[]; flat=[]
    schema_cache={}
    for r in rows:
        db=r['db_id']
        if db not in schema_cache: schema_cache[db]=spider_schema_text(db)
        schema=schema_cache[db]
        example_id=hashlib.sha256(f"{db}\0{r['question']}\0{r['gold_sql']}".encode()).hexdigest()[:20]
        chats.append({'id':example_id,'db_id':db,'source':r['origin'],'messages':make_messages(r['question'],schema,r['gold_sql'])})
        flat.append({'id':example_id,'db_id':db,'origin':r['origin'],'question':r['question'],'schema':schema,'gold_sql':r['gold_sql']})
    return chats,flat
train_chat,train_flat=add_formats(train_rows)
val_chat,val_flat=add_formats(val_rows)
print('First chat sample:')
print(json.dumps(train_chat[0],indent=2,ensure_ascii=False)[:1800])


First chat sample:
{
  "id": "89f2ba095ce47e927120",
  "db_id": "department_management",
  "source": "train_spider",
  "messages": [
    {
      "role": "system",
      "content": "You are an expert text-to-SQL assistant. Generate exactly one valid SQLite SELECT query. Use only the tables and columns in the schema. Return SQL only, without markdown or explanation."
    },
    {
      "role": "user",
      "content": "Schema:\nTable: department\n  Department_ID number\n  Name text\n  Creation text\n  Ranking number\n  Budget_in_Billions number\n  Num_Employees number\n\nTable: head\n  head_ID number\n  name text\n  born_state text\n  age number\n\nTable: management\n  department_ID number\n  head_ID number\n  temporary_acting text\n\n\nForeign keys:\n\nmanagement.head_ID = head.head_ID\n\nmanagement.department_ID = department.Department_ID\n\nQuestion: How many heads of the departments are older than 56 ?"
    },
    {
      "role": "assistant",
      "content": "SELECT count(*) FROM he

## 11 — Quality and leakage checks

In [11]:
def sample_keys(rows): return {(x['db_id'],x['question'],x['gold_sql']) for x in rows}
assert not (sample_keys(train_flat)&sample_keys(val_flat))
assert not ({r['db_id'] for r in train_flat}&{r['db_id'] for r in val_flat})
assert not ({r['db_id'] for r in train_flat+val_flat}&dev_dbs)
assert all(len(x['messages'])==3 and x['messages'][-1]['role']=='assistant' for x in train_chat+val_chat)
assert all(x['gold_sql'].strip() for x in train_flat+val_flat)
print('PASS — no train/validation DB overlap')
print('PASS — no Spider dev DB overlap')
print('PASS — all examples contain assistant SQL target')
print('PASS — gold query parse/compilation checks completed')


PASS — no train/validation DB overlap
PASS — no Spider dev DB overlap
PASS — all examples contain assistant SQL target
PASS — gold query parse/compilation checks completed


## 12 — Save JSONL/CSV artifacts and split manifest atomically

In [12]:
import os
from datetime import datetime, timezone

def write_jsonl(path,items):
    tmp=path.with_suffix(path.suffix+'.tmp')
    with tmp.open('w',encoding='utf-8') as f:
        for item in items:f.write(json.dumps(item,ensure_ascii=False)+'\n')
    os.replace(tmp,path)

def write_csv(path,rows):
    pd.DataFrame(rows).to_csv(path,index=False)

files={
    'train_chat':OUTPUT/'train_chat.jsonl',
    'val_chat':OUTPUT/'val_chat.jsonl',
    'train_flat':OUTPUT/'train_flat.jsonl',
    'val_flat':OUTPUT/'val_flat.jsonl',
    'excluded':OUTPUT/'excluded_records.csv',
    'gold_quarantine':OUTPUT/'gold_sql_quarantine.csv',
}
for key,rows in [('train_chat',train_chat),('val_chat',val_chat),('train_flat',train_flat),('val_flat',val_flat)]:
    write_jsonl(files[key],rows)
write_csv(files['excluded'],excluded)
write_csv(files['gold_quarantine'],rejected)
manifest={
 'project':'SQLSmith','stage':'Milestone B / Notebook 07','dataset':'Spider 1.0','source_repository':ARCHIVE_REPO,
 'source_archive_sha256':archive_hash,'sources':(['train_spider.json']+(['train_others.json'] if train_extra else [])),
 'seed':SEED,'validation_fraction_of_database_ids':VALIDATION_DB_FRACTION,
 'train_database_ids':sorted(train_dbs),'validation_database_ids':sorted(val_dbs),
 'train_examples':len(train_chat),'validation_examples':len(val_chat),'excluded_count':len(excluded),
 'exact_duplicate_count':len(duplicates),'gold_sql_quarantine_count':len(rejected),
 'gold_sql_rejection_summary':dict(REJECTION_COUNTS),
 'spider_dev_database_ids_excluded':len(dev_dbs),
 'system_prompt':SYSTEM_PROMPT,
 'validation_design':'Database-disjoint internal validation from Spider training pool; untouched Spider dev for external evaluation',
 'format':'Hugging Face chat messages JSONL; assistant target is SQL',
 'utc_generated':datetime.now(timezone.utc).isoformat()
}
manifest['artifact_sha256']={key:sha256_file(path) for key,path in files.items()}
manifest_path=OUTPUT/'dataset_manifest.json'
manifest_path.write_text(json.dumps(manifest,indent=2),encoding='utf-8')
print('Saved dataset to',OUTPUT)
for key,path in files.items(): print(key, path.name, manifest['artifact_sha256'][key][:16])
print('Manifest:',manifest_path)


Saved dataset to /content/drive/MyDrive/Prajwal/Projects/SQLSmith/data/processed/sft_v1
train_chat train_chat.jsonl 8d616419aebed7dd
val_chat val_chat.jsonl 474a0680030b288d
train_flat train_flat.jsonl fc02c94bd823e520
val_flat val_flat.jsonl 85eee50f2a77d9e1
excluded excluded_records.csv 01ba4719c80b6fe9
gold_quarantine gold_sql_quarantine.csv 0e7c3b687eb0fed8
Manifest: /content/drive/MyDrive/Prajwal/Projects/SQLSmith/data/processed/sft_v1/dataset_manifest.json


## 13 — Summarize dataset sizes and report limitations

In [14]:
summary=pd.DataFrame([
 {'split':'SFT training','examples':len(train_chat),'databases':len(train_dbs)},
 {'split':'Internal validation','examples':len(val_chat),'databases':len(val_dbs)},
 {'split':'Spider development (reserved)','examples':len(dev),'databases':len(dev_dbs)}
])
display(summary)
print('Validation examples are for training checkpoint selection; official Spider dev is reserved for model comparison.')
print('Note: training targets are public annotated SQL, not distilled frontier-model outputs.')


,split,examples,databases
0,SFT training,7931,131
1,Internal validation,717,15
2,Spider development (reserved),1034,20


Validation examples are for training checkpoint selection; official Spider dev is reserved for model comparison.
Note: training targets are public annotated SQL, not distilled frontier-model outputs.
